# Pump test analysis

Commissioning check for pump flow versus VFD output frequency.

This notebook intentionally keeps the fitted model on the same basis as the bucket-style water commissioning test and the digitized vendor water curve: approximately zero differential pressure. The vendor HFE values in `analysis/docs/pump.pdf` are treated as an operating guarantee envelope, not as paired curve points.

Key assumptions:

- Bottom x-axis: VFD output frequency.
- Top x-axis: estimated shaft speed using `rpm = 30 * Hz`.
- Water vendor curve: digitized from the Series 2, 8 mm / 12 teeth, water 20 C, 1 mPa s chart at approximately zero differential pressure.
- HFE means HFE-7200. At the selected zero-pressure basis, the gear-pump slip term is zero, so no separate HFE curve is plotted; HFE uses the water vendor displacement curve at this basis.
- HFE recirculation data are shown only as a verification note; they are not used to fit or tune the plotted model.


In [1]:
#!/usr/bin/env python3
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.optimize import curve_fit

# ===== Constants and plotting config =====
RPM_PER_HZ = 30.0
PUMP_MAX_FREQ_HZ = 71.7
PUMP_MAX_RPM = PUMP_MAX_FREQ_HZ * RPM_PER_HZ
VENDOR_WATER_DIGITIZATION_SIGMA_LMIN = 0.15  # fixed 1-sigma digitization uncertainty, not a percent-of-flow model

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 14,
    'axes.labelsize': 17,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 13,
    'axes.grid': True,
    'grid.alpha': 0.28,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'analysis' / 'docs' / 'pump.pdf').exists() and (candidate / 'data').exists():
            return candidate
    raise RuntimeError('Could not find repository root from current working directory.')


REPO_ROOT = find_repo_root()
print(f'Repo root: {REPO_ROOT}')
print('Model basis: commissioning / vendor-water zero differential pressure approximation.')
print('HFE-7200 recirculation measurements are verification-only and are not used in any fit.\n')

# ===== Data =====
# Measured water points from 500 mL timing during commissioning.
f_meas_hz = np.array([10.0, 20.0, 30.0, 40.0])
t_meas_s = np.array([28.82, 14.52, 9.77, 7.37])
volume_l = 0.500
sigma_volume_l = 0.05 * volume_l
sigma_time_s = 0.5
Q_meas_lmin = 60.0 * volume_l / t_meas_s
sigma_Q_meas_lmin = np.sqrt(
    (60.0 / t_meas_s) ** 2 * sigma_volume_l**2
    + (60.0 * volume_l / t_meas_s**2) ** 2 * sigma_time_s**2
)

# Vendor water curve, digitized from analysis/docs/pump.pdf page 2:
# Series 2, 8 mm / 12 teeth, water 20 C, 1 mPa s, approximately delta P = 0 bar.
vendor_water_rpm_Q = np.array([
    [200.0, 0.55],
    [300.0, 1.00],
    [500.0, 1.60],
    [750.0, 2.40],
    [1000.0, 3.10],
    [1350.0, 4.30],
    [1500.0, 4.80],
    [2000.0, 6.20],
    [2500.0, 7.90],
    [3000.0, 9.40],
], dtype=float)
f_vendor_hz = vendor_water_rpm_Q[:, 0] / RPM_PER_HZ
Q_vendor_lmin = vendor_water_rpm_Q[:, 1]
sigma_vendor_lmin = np.full_like(Q_vendor_lmin, VENDOR_WATER_DIGITIZATION_SIGMA_LMIN)

# Vendor HFE application guarantee from analysis/docs/pump.pdf page 1.
# These are independent min/max guarantee values for the application envelope, not paired curve points.
hfe_guarantee = {
    'speed_min_rpm': 200.0,
    'speed_max_rpm': 2150.0,
    'flow_min_lmin': 0.5,
    'flow_max_lmin': 4.0,
    'temperature_range_C': '-100 to +30',
    'viscosity_range_mPa_s': '0.39 to 39.0',
    'differential_pressure_bar': '2 to 10 maximum',
}
hfe_speed_min_hz = hfe_guarantee['speed_min_rpm'] / RPM_PER_HZ
hfe_speed_max_hz = hfe_guarantee['speed_max_rpm'] / RPM_PER_HZ

# ===== Fits =====
def fit_origin_wls_with_covariance(f_hz, Q_lmin, sigma_lmin):
    """Origin-constrained WLS fit Q = m f with a covariance-based 1-sigma slope band."""
    popt, pcov = curve_fit(
        lambda x, m: m * x,
        f_hz,
        Q_lmin,
        sigma=sigma_lmin,
        absolute_sigma=True,
    )
    slope = float(popt[0])
    slope_sigma = float(np.sqrt(pcov[0, 0]))
    residual = Q_lmin - slope * f_hz
    chi2 = float(np.sum((residual / sigma_lmin) ** 2))
    dof = int(len(f_hz) - 1)
    return slope, slope_sigma, chi2, dof


def fit_free_line_check(f_hz, Q_lmin, sigma_lmin):
    """Free two-parameter WLS check. The plotted model still uses the origin-constrained fit."""
    popt, pcov = curve_fit(
        lambda x, a, b: a * x + b,
        f_hz,
        Q_lmin,
        sigma=sigma_lmin,
        absolute_sigma=True,
    )
    a, b = map(float, popt)
    sigma_a, sigma_b = np.sqrt(np.diag(pcov))
    residual = Q_lmin - (a * f_hz + b)
    chi2 = float(np.sum((residual / sigma_lmin) ** 2))
    dof = int(len(f_hz) - 2)
    return a, b, float(sigma_a), float(sigma_b), chi2, dof


m_meas, sigma_m_meas, chi2_meas, dof_meas = fit_origin_wls_with_covariance(
    f_meas_hz,
    Q_meas_lmin,
    sigma_Q_meas_lmin,
)
m_vendor, sigma_m_vendor, chi2_vendor, dof_vendor = fit_origin_wls_with_covariance(
    f_vendor_hz,
    Q_vendor_lmin,
    sigma_vendor_lmin,
)

f_grid_hz = np.linspace(0.0, PUMP_MAX_FREQ_HZ, 600)
Q_meas_fit = m_meas * f_grid_hz
Q_meas_low = (m_meas - sigma_m_meas) * f_grid_hz
Q_meas_high = (m_meas + sigma_m_meas) * f_grid_hz

Q_vendor_fit = m_vendor * f_grid_hz
Q_vendor_low = (m_vendor - sigma_m_vendor) * f_grid_hz
Q_vendor_high = (m_vendor + sigma_m_vendor) * f_grid_hz

# At delta P = 0, a gear-pump slip correction K * deltaP / viscosity is zero,
# so HFE-7200 uses the same plotted displacement curve as the vendor water fit.

# ===== HFE-7200 property context for the zero-pressure extrapolation =====
hfe7200_reference_T_C = np.array([25.0, 0.0, -10.0, -20.0, -30.0, -40.0, -50.0, -60.0, -70.0, -100.0, -120.0])
hfe7200_reference_nu_cSt = np.array([0.41, 0.67, 0.78, 0.93, 1.14, 1.42, 1.84, 2.48, 3.72, 12.47, 64.47])
order = np.argsort(hfe7200_reference_T_C)
hfe7200_nu_interp = PchipInterpolator(
    hfe7200_reference_T_C[order],
    np.log(hfe7200_reference_nu_cSt[order]),
    extrapolate=True,
)


def hfe7200_density_kg_m3(temperature_C):
    return 1000.0 * (1.4811 - 0.0023026 * np.asarray(temperature_C, dtype=float))


def hfe7200_kinematic_viscosity_cSt(temperature_C):
    return np.exp(hfe7200_nu_interp(np.asarray(temperature_C, dtype=float)))


hfe_property_table = pd.DataFrame({'temperature_C': [25.0, -110.0]})
hfe_property_table['density_kg_m3'] = hfe7200_density_kg_m3(hfe_property_table['temperature_C'])
hfe_property_table['kinematic_viscosity_cSt'] = hfe7200_kinematic_viscosity_cSt(hfe_property_table['temperature_C'])
hfe_property_table['dynamic_viscosity_mPa_s'] = (
    hfe_property_table['kinematic_viscosity_cSt'] * hfe_property_table['density_kg_m3'] / 1000.0
)
hfe_property_table['zero_dp_slip_correction_lmin'] = 0.0

# ===== Console fit checks =====
def report_free_fit(name, f_hz, Q_lmin, sigma_lmin):
    a, b, sigma_a, sigma_b, chi2, dof = fit_free_line_check(f_hz, Q_lmin, sigma_lmin)
    z_intercept = b / sigma_b if sigma_b > 0.0 else np.inf
    compatible = 'YES' if abs(z_intercept) <= 2.0 else 'NO'
    print(f'{name} free-fit check: Q = a f + b')
    print(f'  a = {a:.6f} +/- {sigma_a:.6f} L/min/Hz')
    print(f'  b = {b:.6f} +/- {sigma_b:.6f} L/min ({z_intercept:+.2f} sigma from zero; origin-compatible: {compatible})')
    print(f'  chi2/dof = {chi2:.2f}/{dof}\n')


report_free_fit('Water measured', f_meas_hz, Q_meas_lmin, sigma_Q_meas_lmin)
report_free_fit('Water vendor digitized', f_vendor_hz, Q_vendor_lmin, sigma_vendor_lmin)
print('Vendor HFE guarantee: no fit is performed; min/max values are treated as independent envelope bounds.\n')

fit_summary = pd.DataFrame([
    {
        'curve': 'water measured commissioning',
        'basis': '500 mL timing; propagated volume/time 1-sigma',
        'slope_L_min_per_Hz': m_meas,
        'slope_sigma_L_min_per_Hz': sigma_m_meas,
        'displacement_mL_per_rev': 1000.0 * m_meas / RPM_PER_HZ,
        'displacement_sigma_mL_per_rev': 1000.0 * sigma_m_meas / RPM_PER_HZ,
        'chi2_dof': f'{chi2_meas:.2f}/{dof_meas}',
    },
    {
        'curve': 'water vendor digitized',
        'basis': f'water 20 C, 1 mPa s, approx 0 bar; fixed digitization sigma={VENDOR_WATER_DIGITIZATION_SIGMA_LMIN:.2f} L/min',
        'slope_L_min_per_Hz': m_vendor,
        'slope_sigma_L_min_per_Hz': sigma_m_vendor,
        'displacement_mL_per_rev': 1000.0 * m_vendor / RPM_PER_HZ,
        'displacement_sigma_mL_per_rev': 1000.0 * sigma_m_vendor / RPM_PER_HZ,
        'chi2_dof': f'{chi2_vendor:.2f}/{dof_vendor}',
    },
])

print('Origin-constrained plotted equations, Q [L/min] = m * f_VFD [Hz]:')
for _, row in fit_summary.iterrows():
    print(
        f"  {row['curve']}: m = {row['slope_L_min_per_Hz']:.5f} +/- "
        f"{row['slope_sigma_L_min_per_Hz']:.5f}; "
        f"D = {row['displacement_mL_per_rev']:.3f} +/- {row['displacement_sigma_mL_per_rev']:.3f} mL/rev"
    )
print('HFE-7200 at delta P=0 uses the same water vendor displacement curve; no separate HFE curve is plotted.')
print(f'Axis check: {PUMP_MAX_FREQ_HZ:.1f} Hz * {RPM_PER_HZ:.1f} rpm/Hz = {PUMP_MAX_RPM:.0f} rpm.\n')

# ===== Verification-only recirculation notes =====
verification_table = pd.DataFrame([
    {
        'source': 'pump_performance.ipynb',
        'check': 'HFE recirculation flow-speed slopes across temperature bins',
        'value': '3.0-3.2 mL/rev',
        'comparison': f'water vendor zero-dP displacement = {1000.0 * m_vendor / RPM_PER_HZ:.2f} mL/rev',
        'model_use': 'verification only; not fitted',
    },
    {
        'source': 'log_20260422_143345_review.ipynb',
        'check': 'cold slowdown linearity versus actual pump frequency',
        'value': '80% command is +0.08% from the 20/40% frequency-flow reference',
        'comparison': 'supports near-linear flow with actual VFD frequency',
        'model_use': 'verification only; not fitted',
    },
    {
        'source': 'log_20260424_153546_review.ipynb',
        'check': 'cold 40 to 60% ramp delivered volume per revolution',
        'value': '1.627 -> 1.610 mL/rev, -1.1%',
        'comparison': 'does not show a large cold HFE slip penalty over the measured ramp',
        'model_use': 'verification only; not fitted',
    },
])

# ===== Plot =====
fig, ax = plt.subplots(figsize=(9.8, 6.2), constrained_layout=True)

# HFE vendor guarantee envelope first so it stays in the background.
guarantee_rect = Rectangle(
    (hfe_speed_min_hz, hfe_guarantee['flow_min_lmin']),
    hfe_speed_max_hz - hfe_speed_min_hz,
    hfe_guarantee['flow_max_lmin'] - hfe_guarantee['flow_min_lmin'],
    facecolor='tab:orange',
    edgecolor='tab:orange',
    alpha=0.12,
    linewidth=1.3,
    label='_nolegend_',
)
ax.add_patch(guarantee_rect)
# Measured water commissioning fit.
ax.errorbar(
    f_meas_hz,
    Q_meas_lmin,
    yerr=sigma_Q_meas_lmin,
    fmt='o',
    capsize=4,
    markersize=6.5,
    elinewidth=1.5,
    capthick=1.5,
    color='tab:blue',
    ecolor='tab:blue',
    label='_nolegend_',
)
ax.plot(f_grid_hz, Q_meas_fit, '--', color='tab:blue', lw=2.3, label='_nolegend_')
ax.fill_between(f_grid_hz, Q_meas_low, Q_meas_high, color='tab:blue', alpha=0.14, linewidth=0)

# Vendor water fit and digitized points.
ax.plot(
    f_vendor_hz,
    Q_vendor_lmin,
    linestyle='none',
    marker='x',
    markersize=7.5,
    markeredgewidth=1.8,
    color='tab:green',
    alpha=0.92,
    label='_nolegend_',
)
ax.plot(f_grid_hz, Q_vendor_fit, '-', color='tab:green', lw=2.6, label='_nolegend_')
ax.fill_between(f_grid_hz, Q_vendor_low, Q_vendor_high, color='tab:green', alpha=0.14, linewidth=0)
ax.set_xlabel('VFD output frequency [Hz]', labelpad=8)
ax.set_ylabel('Volume flow [L/min]', labelpad=8)
ax.set_xlim(0.0, PUMP_MAX_FREQ_HZ)
ax.set_ylim(0.0, 7.8)
ax.tick_params(axis='both', which='major', labelsize=15)

secax = ax.secondary_xaxis('top', functions=(lambda f: f * RPM_PER_HZ, lambda rpm: rpm / RPM_PER_HZ))
secax.set_xlabel('Estimated pump speed [rpm]', labelpad=8)
secax.set_xticks(np.arange(0.0, 2001.0, 500.0))
secax.tick_params(axis='x', which='major', labelsize=15)

legend_handles = [
    Rectangle((0.0, 0.0), 1.0, 1.0, facecolor='tab:orange', edgecolor='tab:orange', alpha=0.12, linewidth=1.3, label='HFE guarantee envelope'),
    Line2D([0], [0], color='tab:blue', lw=2.3, ls='--', marker='o', markersize=7.0, label='Water measured'),
    Rectangle((0.0, 0.0), 1.0, 1.0, facecolor='tab:blue', edgecolor='none', alpha=0.14, label=r'Fit band ($\pm 1\sigma$ slope)'),
    Line2D([0], [0], color='tab:green', lw=2.6, ls='-', marker='x', markersize=8.0, markeredgewidth=1.8, label='Vendor water nominal curve'),
]
ax.legend(handles=legend_handles, loc='upper left', fontsize=13.5, framealpha=0.94, handlelength=2.8)

plt.show()

print('\nVendor HFE guarantee values, shown as an envelope rather than fitted points:')
display(pd.DataFrame([hfe_guarantee]))

print('\nHFE-7200 property context: same zero-pressure displacement curve as water vendor; no separate HFE curve plotted.')
display(hfe_property_table.round({
    'temperature_C': 1,
    'density_kg_m3': 1,
    'kinematic_viscosity_cSt': 3,
    'dynamic_viscosity_mPa_s': 3,
    'zero_dp_slip_correction_lmin': 3,
}))

print('\nFit summary:')
display(fit_summary.round({
    'slope_L_min_per_Hz': 5,
    'slope_sigma_L_min_per_Hz': 5,
    'displacement_mL_per_rev': 3,
    'displacement_sigma_mL_per_rev': 3,
}))

print('\nVerification-only checks from recirculation notebooks:')
display(verification_table)

Repo root: /home/aamy/Documents/hfe-system
Model basis: commissioning / vendor-water zero differential pressure approximation.
HFE-7200 recirculation measurements are verification-only and are not used in any fit.

Water measured free-fit check: Q = a f + b
  a = 0.101546 +/- 0.007401 L/min/Hz
  b = 0.026550 +/- 0.108412 L/min (+0.24 sigma from zero; origin-compatible: YES)
  chi2/dof = 0.01/2

Water vendor digitized free-fit check: Q = a f + b
  a = 0.094129 +/- 0.001581 L/min/Hz
  b = 0.014681 +/- 0.083753 L/min (+0.18 sigma from zero; origin-compatible: YES)
  chi2/dof = 1.50/8

Vendor HFE guarantee: no fit is performed; min/max values are treated as independent envelope bounds.

Origin-constrained plotted equations, Q [L/min] = m * f_VFD [Hz]:
  water measured commissioning: m = 0.10317 +/- 0.00332; D = 3.439 +/- 0.111 mL/rev
  water vendor digitized: m = 0.09436 +/- 0.00090; D = 3.145 +/- 0.030 mL/rev
HFE-7200 at delta P=0 uses the same water vendor displacement curve; no separate

/tmp/ipykernel_1223387/60219503.py:314: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,speed_min_rpm,speed_max_rpm,flow_min_lmin,flow_max_lmin,temperature_range_C,viscosity_range_mPa_s,differential_pressure_bar
0,200.0,2150.0,0.5,4.0,-100 to +30,0.39 to 39.0,2 to 10 maximum



HFE-7200 property context: same zero-pressure displacement curve as water vendor; no separate HFE curve plotted.


,temperature_C,density_kg_m3,kinematic_viscosity_cSt,dynamic_viscosity_mPa_s,zero_dp_slip_correction_lmin
0,25.0,1423.5,0.410,0.584,0.0
1,-110.0,1734.4,25.431,44.107,0.0



Fit summary:


,curve,basis,slope_L_min_per_Hz,slope_sigma_L_min_per_Hz,displacement_mL_per_rev,displacement_sigma_mL_per_rev,chi2_dof
0,water measured commissioning,500 mL timing; propagated volume/time 1-sigma,0.10317,0.00332,3.439,0.111,0.07/3
1,water vendor digitized,"water 20 C, 1 mPa s, approx 0 bar; fixed digit...",0.09436,0.00090,3.145,0.030,1.53/9



Verification-only checks from recirculation notebooks:


,source,check,value,comparison,model_use
0,pump_performance.ipynb,HFE recirculation flow-speed slopes across tem...,3.0-3.2 mL/rev,water vendor zero-dP displacement = 3.15 mL/rev,verification only; not fitted
1,log_20260422_143345_review.ipynb,cold slowdown linearity versus actual pump fre...,80% command is +0.08% from the 20/40% frequenc...,supports near-linear flow with actual VFD freq...,verification only; not fitted
2,log_20260424_153546_review.ipynb,cold 40 to 60% ramp delivered volume per revol...,"1.627 -> 1.610 mL/rev, -1.1%",does not show a large cold HFE slip penalty ov...,verification only; not fitted


In [2]:
# Volume-per-revolution comparison for the plotted origin-constrained curves.
D_meas_mL_per_rev = 1000.0 * m_meas / RPM_PER_HZ
sigma_D_meas_mL_per_rev = 1000.0 * sigma_m_meas / RPM_PER_HZ
D_vendor_mL_per_rev = 1000.0 * m_vendor / RPM_PER_HZ
sigma_D_vendor_mL_per_rev = 1000.0 * sigma_m_vendor / RPM_PER_HZ
delta_D_mL_per_rev = D_meas_mL_per_rev - D_vendor_mL_per_rev
sigma_delta_D_mL_per_rev = np.sqrt(sigma_D_meas_mL_per_rev**2 + sigma_D_vendor_mL_per_rev**2)
delta_D_percent = 100.0 * delta_D_mL_per_rev / D_vendor_mL_per_rev
delta_D_sigma = delta_D_mL_per_rev / sigma_delta_D_mL_per_rev

displacement_comparison = pd.DataFrame([
    {
        'curve': 'Water measured',
        'basis': 'commissioning 500 mL timing',
        'displacement_mL_per_rev': D_meas_mL_per_rev,
        'one_sigma_mL_per_rev': sigma_D_meas_mL_per_rev,
        'difference_vs_vendor_mL_per_rev': delta_D_mL_per_rev,
        'difference_vs_vendor_percent': delta_D_percent,
        'difference_sigma': delta_D_sigma,
    },
    {
        'curve': 'Vendor water nominal curve',
        'basis': 'digitized pump.pdf water curve, approx 0 bar',
        'displacement_mL_per_rev': D_vendor_mL_per_rev,
        'one_sigma_mL_per_rev': sigma_D_vendor_mL_per_rev,
        'difference_vs_vendor_mL_per_rev': 0.0,
        'difference_vs_vendor_percent': 0.0,
        'difference_sigma': 0.0,
    },
])

print('Volume per revolution from origin-constrained fits:')
print(f'  Water measured: {D_meas_mL_per_rev:.3f} +/- {sigma_D_meas_mL_per_rev:.3f} mL/rev')
print(f'  Vendor water nominal curve: {D_vendor_mL_per_rev:.3f} +/- {sigma_D_vendor_mL_per_rev:.3f} mL/rev')
print(
    f'  Difference: {delta_D_mL_per_rev:+.3f} +/- {sigma_delta_D_mL_per_rev:.3f} mL/rev '
    f'({delta_D_percent:+.1f}%, {delta_D_sigma:+.1f} sigma using plotted uncertainties)'
)
print('  Vendor uncertainty here is digitization-only; no vendor tolerance envelope is specified in pump.pdf.')
display(displacement_comparison.round({
    'displacement_mL_per_rev': 3,
    'one_sigma_mL_per_rev': 3,
    'difference_vs_vendor_mL_per_rev': 3,
    'difference_vs_vendor_percent': 1,
    'difference_sigma': 1,
}))


Volume per revolution from origin-constrained fits:
  Water measured: 3.439 +/- 0.111 mL/rev
  Vendor water nominal curve: 3.145 +/- 0.030 mL/rev
  Difference: +0.294 +/- 0.115 mL/rev (+9.3%, +2.6 sigma using plotted uncertainties)
  Vendor uncertainty here is digitization-only; no vendor tolerance envelope is specified in pump.pdf.


,curve,basis,displacement_mL_per_rev,one_sigma_mL_per_rev,difference_vs_vendor_mL_per_rev,difference_vs_vendor_percent,difference_sigma
0,Water measured,commissioning 500 mL timing,3.439,0.111,0.294,9.3,2.6
1,Vendor water nominal curve,"digitized pump.pdf water curve, approx 0 bar",3.145,0.030,0.000,0.0,0.0


In [3]:
# Full vendor water performance curve digitized from analysis/docs/pump.pdf page 2.
# The original chart gives straight constant-speed curves for water at 20 C and 1 mPa s.
# These endpoint values are digitized from the scanned chart and should be treated as
# graphical data, not vendor-tabulated numbers.
vendor_water_full_endpoints = pd.DataFrame([
    {'speed_rpm': 200.0,  'p1_bar': 0.0, 'q1_lmin': 0.55, 'p2_bar': 2.7,  'q2_lmin': 0.00},
    {'speed_rpm': 300.0,  'p1_bar': 0.0, 'q1_lmin': 1.00, 'p2_bar': 4.2,  'q2_lmin': 0.00},
    {'speed_rpm': 500.0,  'p1_bar': 0.0, 'q1_lmin': 1.60, 'p2_bar': 8.8,  'q2_lmin': 0.00},
    {'speed_rpm': 750.0,  'p1_bar': 0.0, 'q1_lmin': 2.40, 'p2_bar': 16.8, 'q2_lmin': 0.00},
    {'speed_rpm': 1000.0, 'p1_bar': 0.0, 'q1_lmin': 3.10, 'p2_bar': 20.0, 'q2_lmin': 0.25},
    {'speed_rpm': 1350.0, 'p1_bar': 0.0, 'q1_lmin': 4.30, 'p2_bar': 20.0, 'q2_lmin': 1.20},
    {'speed_rpm': 1500.0, 'p1_bar': 0.0, 'q1_lmin': 4.80, 'p2_bar': 20.0, 'q2_lmin': 1.65},
    {'speed_rpm': 2000.0, 'p1_bar': 0.0, 'q1_lmin': 6.20, 'p2_bar': 20.0, 'q2_lmin': 3.30},
    {'speed_rpm': 2500.0, 'p1_bar': 0.0, 'q1_lmin': 7.90, 'p2_bar': 20.0, 'q2_lmin': 4.70},
    {'speed_rpm': 3000.0, 'p1_bar': 0.0, 'q1_lmin': 9.40, 'p2_bar': 20.0, 'q2_lmin': 6.40},
])
vendor_water_full_endpoints['slope_L_min_per_bar'] = (
    (vendor_water_full_endpoints['q2_lmin'] - vendor_water_full_endpoints['q1_lmin'])
    / (vendor_water_full_endpoints['p2_bar'] - vendor_water_full_endpoints['p1_bar'])
)
vendor_water_full_endpoints['zero_pressure_displacement_mL_per_rev'] = (
    1000.0 * vendor_water_full_endpoints['q1_lmin'] / vendor_water_full_endpoints['speed_rpm']
)
vendor_water_full_endpoints['zero_pressure_frequency_Hz'] = vendor_water_full_endpoints['speed_rpm'] / RPM_PER_HZ

pressure_grid_bar = np.arange(0.0, 20.0 + 1e-9, 2.0)
records = []
for row in vendor_water_full_endpoints.itertuples(index=False):
    for pressure_bar in pressure_grid_bar:
        q_lmin = row.q1_lmin + row.slope_L_min_per_bar * (pressure_bar - row.p1_bar)
        records.append({
            'speed_rpm': row.speed_rpm,
            'frequency_Hz': row.speed_rpm / RPM_PER_HZ,
            'differential_pressure_bar': pressure_bar,
            'flow_lmin': q_lmin if q_lmin >= -0.03 else np.nan,
        })
vendor_water_full_digitized = pd.DataFrame.from_records(records)

vendor_curve_csv = REPO_ROOT / 'analysis' / 'notebooks' / 'vendor_water_performance_digitized.csv'
vendor_water_full_digitized.to_csv(vendor_curve_csv, index=False)

fig, ax = plt.subplots(figsize=(9.8, 6.2), constrained_layout=True)
ax.set_xlim(0.0, 20.0)
ax.set_ylim(0.0, 10.1)


def line_angle_degrees(axis, x_start, y_start, x_end, y_end):
    """Return the line angle in display coordinates so labels follow plotted slopes."""
    start_px = axis.transData.transform((x_start, y_start))
    end_px = axis.transData.transform((x_end, y_end))
    return np.degrees(np.arctan2(end_px[1] - start_px[1], end_px[0] - start_px[0]))


vendor_curve_color_values = plt.cm.Blues(np.linspace(0.55, 0.95, len(vendor_water_full_endpoints)))
vendor_curve_colors = dict(zip(vendor_water_full_endpoints['speed_rpm'], vendor_curve_color_values))
rpm_label_specs = {
    200.0: {'p_bar': 0.28, 'dy_lmin': 0.07, 'ha': 'left', 'va': 'bottom'},
    300.0: {'p_bar': 2.05, 'dy_lmin': 0.13, 'ha': 'left', 'va': 'bottom'},
    500.0: {'p_bar': 4.40, 'dy_lmin': 0.13, 'ha': 'left', 'va': 'bottom'},
    750.0: {'p_bar': 8.90, 'dy_lmin': 0.13, 'ha': 'left', 'va': 'bottom'},
    1000.0: {'p_bar': 17.75, 'dy_lmin': 0.10, 'ha': 'left', 'va': 'bottom'},
    1350.0: {'p_bar': 17.75, 'dy_lmin': 0.10, 'ha': 'left', 'va': 'bottom'},
    1500.0: {'p_bar': 17.75, 'dy_lmin': 0.10, 'ha': 'left', 'va': 'bottom'},
    2000.0: {'p_bar': 17.75, 'dy_lmin': 0.10, 'ha': 'left', 'va': 'bottom'},
    2500.0: {'p_bar': 17.75, 'dy_lmin': 0.10, 'ha': 'left', 'va': 'bottom'},
    3000.0: {'p_bar': 17.75, 'dy_lmin': 0.10, 'ha': 'left', 'va': 'bottom'},
}

for row in vendor_water_full_endpoints.itertuples(index=False):
    color = vendor_curve_colors[row.speed_rpm]
    p_stop = min(20.0, row.p2_bar if row.q2_lmin <= 0.0 else 20.0)
    p_line = np.linspace(row.p1_bar, p_stop, 160)
    q_line = row.q1_lmin + row.slope_L_min_per_bar * (p_line - row.p1_bar)
    ax.plot(p_line, q_line, color=color, lw=2.1)

    label_spec = rpm_label_specs[row.speed_rpm]
    label_p = min(label_spec['p_bar'], p_stop - 0.15)
    ha = label_spec['ha']
    label_q = row.q1_lmin + row.slope_L_min_per_bar * (label_p - row.p1_bar)
    label_angle = line_angle_degrees(ax, row.p1_bar, row.q1_lmin, row.p2_bar, row.q2_lmin)
    ax.text(
        label_p,
        max(label_q + label_spec['dy_lmin'], 0.16),
        f'{row.speed_rpm:.0f} rpm',
        color=color,
        fontsize=10.5,
        rotation=label_angle,
        rotation_mode='anchor',
        ha=ha,
        va=label_spec['va'],
        bbox={'boxstyle': 'round,pad=0.12', 'fc': 'white', 'ec': 'none', 'alpha': 0.72},
    )

ax.set_xlabel('Differential pressure [bar]', labelpad=8)
ax.set_ylabel('Volume flow [L/min]', labelpad=8)
ax.set_xlim(0.0, 20.0)
ax.set_ylim(0.0, 10.1)
ax.set_xticks(np.arange(0.0, 20.1, 2.0))
ax.set_xticks(np.arange(0.0, 20.01, 0.5), minor=True)
ax.set_yticks(np.arange(0.0, 10.1, 1.0))
ax.set_yticks(np.arange(0.0, 10.01, 0.2), minor=True)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.tick_params(axis='both', which='minor', length=3, color='0.55')
ax.grid(True, which='major', color='0.55', alpha=0.30, linewidth=0.9)
ax.grid(True, which='minor', color='0.70', alpha=0.22, linewidth=0.55, linestyle=':')

plt.show()

print('Digitized full vendor water performance curve from pump.pdf page 2.')
print('Basis: water 20 C, 1 mPa s; Series 2 pump head, 8 mm / 12 teeth.')
print('Digitization uncertainty is graphical; use about +/-0.1 L/min and +/-0.2 bar as a practical reading scale.')
print(f'Digitized table saved to: {vendor_curve_csv}')

print('\nDigitized curve endpoints and slopes:')
display(vendor_water_full_endpoints.round({
    'speed_rpm': 0,
    'p1_bar': 2,
    'q1_lmin': 2,
    'p2_bar': 2,
    'q2_lmin': 2,
    'slope_L_min_per_bar': 3,
    'zero_pressure_displacement_mL_per_rev': 3,
    'zero_pressure_frequency_Hz': 2,
}))

print('\nFlow table from the digitized line representation:')
flow_pivot = vendor_water_full_digitized.pivot_table(
    index='differential_pressure_bar',
    columns='speed_rpm',
    values='flow_lmin',
)
flow_pivot.columns = [f'{col:.0f} rpm' for col in flow_pivot.columns]
display(flow_pivot.round(2))

Digitized full vendor water performance curve from pump.pdf page 2.
Basis: water 20 C, 1 mPa s; Series 2 pump head, 8 mm / 12 teeth.
Digitization uncertainty is graphical; use about +/-0.1 L/min and +/-0.2 bar as a practical reading scale.
Digitized table saved to: /home/aamy/Documents/hfe-system/analysis/notebooks/vendor_water_performance_digitized.csv

Digitized curve endpoints and slopes:


/tmp/ipykernel_1223387/2883509259.py:107: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,speed_rpm,p1_bar,q1_lmin,p2_bar,q2_lmin,slope_L_min_per_bar,zero_pressure_displacement_mL_per_rev,zero_pressure_frequency_Hz
0,200.0,0.0,0.55,2.7,0.00,-0.204,2.750,6.67
1,300.0,0.0,1.00,4.2,0.00,-0.238,3.333,10.00
2,500.0,0.0,1.60,8.8,0.00,-0.182,3.200,16.67
3,750.0,0.0,2.40,16.8,0.00,-0.143,3.200,25.00
4,1000.0,0.0,3.10,20.0,0.25,-0.143,3.100,33.33
5,1350.0,0.0,4.30,20.0,1.20,-0.155,3.185,45.00
6,1500.0,0.0,4.80,20.0,1.65,-0.158,3.200,50.00
7,2000.0,0.0,6.20,20.0,3.30,-0.145,3.100,66.67
8,2500.0,0.0,7.90,20.0,4.70,-0.160,3.160,83.33
9,3000.0,0.0,9.40,20.0,6.40,-0.150,3.133,100.00



Flow table from the digitized line representation:


,200 rpm,300 rpm,500 rpm,750 rpm,1000 rpm,1350 rpm,1500 rpm,2000 rpm,2500 rpm,3000 rpm
differential_pressure_bar,,,,,,,,,,
0.0,0.55,1.00,1.60,2.40,3.10,4.30,4.80,6.20,7.90,9.4
2.0,0.14,0.52,1.24,2.11,2.82,3.99,4.48,5.91,7.58,9.1
4.0,NaN,0.05,0.87,1.83,2.53,3.68,4.17,5.62,7.26,8.8
6.0,NaN,NaN,0.51,1.54,2.24,3.37,3.85,5.33,6.94,8.5
8.0,NaN,NaN,0.15,1.26,1.96,3.06,3.54,5.04,6.62,8.2
10.0,NaN,NaN,NaN,0.97,1.67,2.75,3.22,4.75,6.30,7.9
12.0,NaN,NaN,NaN,0.69,1.39,2.44,2.91,4.46,5.98,7.6
14.0,NaN,NaN,NaN,0.40,1.10,2.13,2.60,4.17,5.66,7.3
16.0,NaN,NaN,NaN,0.11,0.82,1.82,2.28,3.88,5.34,7.0
